In [1]:
import duckdb
con = duckdb.connect('nyc311.db')

# Load the income CSV as a new table
con.execute("""
    CREATE OR REPLACE TABLE income AS 
    SELECT * FROM read_csv_auto('data/all_zcta_income.csv')
""")

con.execute("SELECT COUNT(*) FROM income").fetchdf()

,count_star()
0,33774


In [2]:
con.execute("""
    SELECT zcta, median_income 
    FROM income 
    WHERE median_income = '-666666666'
""").fetchdf()

,zcta,median_income
0,00636,-666666666
1,00694,-666666666
2,00786,-666666666
3,00906,-666666666
4,00934,-666666666
...,...,...
3148,99825,-666666666
3149,99850,-666666666
3150,99903,-666666666
3151,99923,-666666666


In [3]:
con.execute("""
    CREATE OR REPLACE TABLE income_clean AS
    SELECT 
        zcta,
        CAST(median_income AS INTEGER) AS median_income
    FROM income
    WHERE median_income != '-666666666'
""")

# Now check how well the join actually matches
con.execute("""
    SELECT 
        COUNT(DISTINCT r.incident_zip) AS total_zips_in_311,
        COUNT(DISTINCT i.zcta) AS matched_zips
    FROM requests r
    LEFT JOIN income_clean i ON r.incident_zip = i.zcta
""").fetchdf()

,total_zips_in_311,matched_zips
0,205,187


In [4]:
con.execute("""
    SELECT DISTINCT r.incident_zip
    FROM requests r
    LEFT JOIN income_clean i ON r.incident_zip = i.zcta
    WHERE i.zcta IS NULL
    ORDER BY r.incident_zip
""").fetchdf()

,incident_zip
0,10000
1,10020
2,10041
3,10112
4,10118
5,10119
6,10121
7,10153
8,10154
9,10168


In [5]:
con.execute("""
    CREATE OR REPLACE TABLE requests_with_income AS
    SELECT 
        r.*,
        i.median_income,
        NTILE(4) OVER (ORDER BY i.median_income) AS income_quartile
    FROM requests r
    JOIN income_clean i ON r.incident_zip = i.zcta
    WHERE r.closed_date IS NOT NULL
""")

con.execute("SELECT COUNT(*) FROM requests_with_income").fetchdf()

,count_star()
0,48742


In [6]:
con.execute("""
    SELECT 
        complaint_type,
        income_quartile,
        MEDIAN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) AS median_hrs,
        COUNT(*) AS n
    FROM requests_with_income
    WHERE complaint_type IN ('HEAT/HOT WATER', 'UNSANITARY CONDITION', 'Illegal Parking')
    GROUP BY complaint_type, income_quartile
    ORDER BY complaint_type, income_quartile
""").fetchdf()

,complaint_type,income_quartile,median_hrs,n
0,HEAT/HOT WATER,1,30.0,1571
1,HEAT/HOT WATER,2,32.0,1253
2,HEAT/HOT WATER,3,26.0,1084
3,HEAT/HOT WATER,4,32.0,436
4,Illegal Parking,1,2.0,1545
5,Illegal Parking,2,2.0,2643
6,Illegal Parking,3,2.0,2766
7,Illegal Parking,4,1.0,2757
8,UNSANITARY CONDITION,1,250.0,720
9,UNSANITARY CONDITION,2,328.0,371


In [7]:
con.execute("""
    SELECT 
        complaint_type,
        income_quartile,
        MEDIAN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) AS median_hrs,
        COUNT(*) AS n
    FROM requests_with_income
    GROUP BY complaint_type, income_quartile
    HAVING COUNT(*) >= 10
    ORDER BY complaint_type, income_quartile
""").fetchdf()

,complaint_type,income_quartile,median_hrs,n
0,APPLIANCE,1,219.5,128
1,APPLIANCE,2,329.0,66
2,APPLIANCE,3,276.0,53
3,APPLIANCE,4,330.0,21
4,Abandoned Bike,4,48.5,20
...,...,...,...,...
296,Water Quality,3,14.0,11
297,Water System,1,7.0,101
298,Water System,2,7.0,136
299,Water System,3,7.0,191


In [8]:
con.execute("""
    WITH quartile_medians AS (
        SELECT complaint_type, income_quartile,
               MEDIAN(DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP))) AS median_hrs,
               COUNT(*) AS n
        FROM requests_with_income
        GROUP BY complaint_type, income_quartile
        HAVING COUNT(*) >= 30
    ),
    pivoted AS (
        SELECT complaint_type,
               MAX(CASE WHEN income_quartile = 1 THEN median_hrs END) AS q1_hrs,
               MAX(CASE WHEN income_quartile = 4 THEN median_hrs END) AS q4_hrs
        FROM quartile_medians
        GROUP BY complaint_type
        HAVING q1_hrs IS NOT NULL AND q4_hrs IS NOT NULL
    )
    SELECT complaint_type, q1_hrs, q4_hrs, 
           ROUND((q1_hrs - q4_hrs) / q4_hrs * 100, 1) AS pct_diff_q1_vs_q4
    FROM pivoted
    ORDER BY pct_diff_q1_vs_q4 DESC
""").fetchdf()

,complaint_type,q1_hrs,q4_hrs,pct_diff_q1_vs_q4
0,Rodent,0.0,0.0,NaN
1,Missed Collection,242.5,50.0,385.0
2,Illegal Dumping,179.0,51.0,251.0
3,Animal-Abuse,3.0,1.0,200.0
4,Graffiti,614.0,221.0,177.8
5,Noise - Residential,2.0,1.0,100.0
6,Noise - Street/Sidewalk,2.0,1.0,100.0
7,Noise - Vehicle,2.0,1.0,100.0
8,Illegal Parking,2.0,1.0,100.0
9,Non-Emergency Police Matter,2.0,1.0,100.0


In [10]:
from scipy import stats

for complaint in ['Missed Collection', 'Illegal Dumping', 'Graffiti', 'WATER LEAK', 
                   'Building/Use', 'PAINT/PLASTER', 'FLOORING/STAIRS', 'General Construction/Plumbing']:
    q1 = con.execute(f"""
        SELECT DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP)) as hrs
        FROM requests_with_income
        WHERE complaint_type = '{complaint}' AND income_quartile = 1
    """).fetchdf()['hrs']
    
    q4 = con.execute(f"""
        SELECT DATEDIFF('hour', CAST(created_date AS TIMESTAMP), CAST(closed_date AS TIMESTAMP)) as hrs
        FROM requests_with_income
        WHERE complaint_type = '{complaint}' AND income_quartile = 4
    """).fetchdf()['hrs']
    
    if len(q1) >= 10 and len(q4) >= 10:
        stat, pval = stats.mannwhitneyu(q1, q4)
        print(f"{complaint}: n1={len(q1)}, n4={len(q4)}, p-value={pval:.4f}")

Missed Collection: n1=100, n4=207, p-value=0.0000
Illegal Dumping: n1=171, n4=157, p-value=0.0000
Graffiti: n1=69, n4=109, p-value=0.0158
WATER LEAK: n1=261, n4=44, p-value=0.0000
Building/Use: n1=51, n4=71, p-value=0.0499
PAINT/PLASTER: n1=379, n4=69, p-value=0.0054
FLOORING/STAIRS: n1=219, n4=34, p-value=0.0317
General Construction/Plumbing: n1=104, n4=179, p-value=0.5447


In [11]:
con.close()